In [14]:
import pandas as pd
import random
import string
import uuid

In [15]:
def gen_text(size=28):
    return str(uuid.uuid4())[0:size]

def gen_integer(minv=1, maxv=9999):
    return random.randint(minv, maxv)

def gen_float(max_v=10.0):
    return random.uniform(0.1, max_v)

def gen_by_option(chars):
    return random.choice(chars)

def gen_ip():
    return ".".join(map(str, (random.randint(0, 255) for _ in range(4))))

def bucket_epss(score):

    if score < 0.2:
        return "< 0.2"
    elif score < 0.4:
        return "< 0.4"
    elif score < 0.6:
        return "< 0.6"
    elif score < 0.8:
        return "< 0.8"
    else:
        return ">= 0.8"

def bucket_cvss(score):
    if score:
        if score < 0.1:
            return "None"
        elif score < 4.0:
            return "low"
        elif score < 7.0:
            return "medium"
        elif score < 9.0:
            return "high"
        else: 
            return "critical"
    else:
        return "None"

In [16]:
df_v1 = pd.DataFrame([
    ["< 0.2",3061,  5448,  39216],
    ["< 0.4",51,    2630,  10243],
    ["< 0.6",35,    3382,  17175],
    ["< 0.8",32,    2936,  13483],
    [">= 0.8",142,  3206,  15316]
], columns=["epss_rank","n_cves","n_orgs","n_ips"])
df_v1

,epss_rank,n_cves,n_orgs,n_ips
0,< 0.2,3061,5448,39216
1,< 0.4,51,2630,10243
2,< 0.6,35,3382,17175
3,< 0.8,32,2936,13483
4,>= 0.8,142,3206,15316


In [17]:
df_v1.to_parquet("df_v1.parquet")

In [18]:
n_size_cves = 100

cves = {}
for _ in range(n_size_cves):
    cve_id = "CVE-%s-%s" % (gen_integer(2019, 2024), gen_text(4))
    epss = round(gen_float(1.0), 2)
    cvss = round(gen_float(10.0), 1)
    b_epss = bucket_epss(epss)
    b_cvss = bucket_cvss(cvss)
    cves[cve_id] = {'epss': epss, 'b_epss': b_epss, 'cvss': cvss, 'b_cvss': b_cvss}

In [19]:
df_v2a = []
n_size_row = 1000


for _ in range(n_size_row):
    cve_id = gen_by_option(list(cves.keys()))
    row = [gen_text(), # org_clean
           gen_ip(), # ip_str
           gen_text(8),  # cpe_product
           str(round(gen_float(5.0), 1)), # cpe_version
           cve_id, # cve_id
           cves[cve_id]['epss'], # epss
           cves[cve_id]['b_epss'], # epss_rank
           cves[cve_id]['cvss'],  # cvss
           cves[cve_id]['b_cvss'], 
           gen_by_option(['2.1', '3.1'])
          ]
    df_v2a.append(row)
    
df_v2a = pd.DataFrame(df_v2a, columns=["org_clean", 'ip_str', 'cpe_product', 'cpe_version', 'cve_id', 'epss', 'epss_rank','cvss','cvss_rank', 'cvss_version'])
df_v2a

,org_clean,ip_str,cpe_product,cpe_version,cve_id,epss,epss_rank,cvss,cvss_rank,cvss_version
0,b5894b27-37a2-48b9-a191-f223,54.5.15.58,c2eeaa61,2.6,CVE-2022-8573,0.96,>= 0.8,0.6,low,2.1
1,31428a48-b44f-439f-995e-6159,182.219.164.31,c83083ba,4.6,CVE-2024-db04,0.88,>= 0.8,1.8,low,2.1
2,f88c9374-bef7-4fc0-82bf-ac72,104.152.164.123,9be8af4c,1.6,CVE-2023-a34c,0.39,< 0.4,3.2,low,2.1
3,6392dbbc-080d-4fe0-ab97-2cd2,200.13.229.1,18b96fd0,1.2,CVE-2023-e4a9,0.94,>= 0.8,0.6,low,2.1
4,331a6532-efe9-4ed8-9fda-e741,116.170.129.165,bdf9c0a6,1.8,CVE-2022-f39b,0.36,< 0.4,6.8,medium,2.1
...,...,...,...,...,...,...,...,...,...,...
995,3cb0114f-b842-40fe-bd98-b1bf,59.228.18.25,43dc1046,1.5,CVE-2023-a6e4,0.61,< 0.8,2.7,low,2.1
996,e14aca30-24f5-4de5-8900-3a18,164.190.66.149,ead65495,2.6,CVE-2019-9d1b,0.77,< 0.8,2.5,low,3.1
997,c10fc3aa-5d85-42ba-b2c2-bbf2,147.221.68.62,ca7c08f7,1.4,CVE-2022-8e14,0.72,< 0.8,8.9,high,2.1
998,1b75a9dc-ab9e-479c-bb13-0efc,140.108.195.45,d6b019f8,2.4,CVE-2022-6836,0.74,< 0.8,5.7,medium,3.1


In [20]:
df_v2a.to_parquet("df_v2a.parquet")

In [21]:
df_v2b = []
n_size_row = 1000

for _ in range(n_size_row):
    
    n_cves = gen_integer(1, 10)
    n_ips = gen_integer(1, 10)
    n_cpes = gen_integer(1, 10)
    
    cve_list = [gen_by_option(list(cves.keys())) for _ in range(n_cves)]
    ips_list = [gen_ip() for _ in range(n_ips)]
    cpe_list = [gen_text(8) for _ in range(n_cpes)]
    
    row = [gen_text(), # org_clean
           cves[cve_id]['epss'], # epss
           cves[cve_id]['b_epss'], # epss_rank
           cve_list,
           ips_list,
           cve_list,
          ]
    df_v2b.append(row)
    
df_v2b = pd.DataFrame(df_v2b, columns=["org_clean", 'epss_major', 'epss_rank_major', 'cpe_list', 'ip_list', 'cve_list'])
df_v2b

,org_clean,epss_major,epss_rank_major,cpe_list,ip_list,cve_list
0,93a32a8b-93ad-4742-8b48-9d69,0.91,>= 0.8,[CVE-2021-5ead],"[54.166.89.111, 87.165.12.106, 0.74.6.208, 108...",[CVE-2021-5ead]
1,19113acf-68dd-494a-8ba2-993e,0.91,>= 0.8,"[CVE-2020-2fd1, CVE-2023-0398, CVE-2024-6952, ...",[148.90.222.201],"[CVE-2020-2fd1, CVE-2023-0398, CVE-2024-6952, ..."
2,70d2d560-0aa1-4c0a-aeab-0034,0.91,>= 0.8,"[CVE-2020-2fd1, CVE-2020-7331, CVE-2021-b958, ...","[189.27.146.172, 124.237.183.168, 21.244.122.5...","[CVE-2020-2fd1, CVE-2020-7331, CVE-2021-b958, ..."
3,9db24c09-4585-466d-bb0b-345a,0.91,>= 0.8,"[CVE-2022-d455, CVE-2022-9c8d]","[224.215.92.246, 180.172.151.38, 190.186.98.21...","[CVE-2022-d455, CVE-2022-9c8d]"
4,e4dd9431-0306-4202-a496-748f,0.91,>= 0.8,"[CVE-2020-2fd1, CVE-2024-a731, CVE-2022-8e14, ...","[239.212.39.236, 12.196.200.133, 82.179.132.23...","[CVE-2020-2fd1, CVE-2024-a731, CVE-2022-8e14, ..."
...,...,...,...,...,...,...
995,184242ee-1e36-4e28-b2d7-00b2,0.91,>= 0.8,"[CVE-2024-db04, CVE-2019-9d1b]","[5.84.180.171, 232.232.255.53, 202.51.227.212,...","[CVE-2024-db04, CVE-2019-9d1b]"
996,783e5ad2-e5c7-4f80-84a6-0749,0.91,>= 0.8,"[CVE-2023-d9b6, CVE-2019-1c71, CVE-2024-c9ee, ...",[209.11.38.67],"[CVE-2023-d9b6, CVE-2019-1c71, CVE-2024-c9ee, ..."
997,1be7b512-ee2e-435c-9ef6-39d6,0.91,>= 0.8,"[CVE-2022-9a39, CVE-2022-11c3, CVE-2019-7e88, ...","[141.234.61.51, 210.2.30.235, 79.137.151.198]","[CVE-2022-9a39, CVE-2022-11c3, CVE-2019-7e88, ..."
998,ece1fd04-0f5b-449a-bae6-e850,0.91,>= 0.8,"[CVE-2022-d455, CVE-2023-a6e4, CVE-2023-b1d6, ...","[223.10.226.33, 229.52.179.240, 152.188.71.159...","[CVE-2022-d455, CVE-2023-a6e4, CVE-2023-b1d6, ..."


In [22]:
df_v2b.to_parquet("df_v2b.parquet")

In [23]:
df_v3 = []
n_size_row = 1000


for _ in range(n_size_row):
    
    cve_id = gen_by_option(list(cves.keys()))
    n_orgs = gen_integer(1, 1000)
    orgs_list = [gen_text(8) for _ in range(n_orgs)]
    
    row = [cves[cve_id]['b_epss'], # epss
           cve_id,
           cves[cve_id]['cvss'],
           cves[cve_id]['b_epss'], # epss_rank
           gen_by_option(['2.1', '3.1']),
           n_orgs,
           gen_integer(2, 100),
           orgs_list
          ]
    df_v3.append(row)
    
df_v3 = pd.DataFrame(df_v3, columns=["epss_rank", 'cve_id', 'cvss','cvss_rank', 'cvss_version', 'n_orgs','n_ips', 'org_list'])
df_v3

,epss_rank,cve_id,cvss,cvss_rank,cvss_version,n_orgs,n_ips,org_list
0,< 0.4,CVE-2022-3ee1,2.5,< 0.4,2.1,644,43,"[b7ea2907, 9c17bbb7, d0158c19, cc902b58, a9016..."
1,< 0.6,CVE-2024-7782,4.3,< 0.6,2.1,678,44,"[4c2f3d9a, 79909bd1, 4ffa1a2b, aa6521ff, 726d8..."
2,< 0.2,CVE-2019-f239,6.6,< 0.2,2.1,452,21,"[823b8b17, 5ae9e1d9, e9c1a8cf, 6cfa0490, cf570..."
3,< 0.8,CVE-2022-8e14,8.9,< 0.8,3.1,368,21,"[b336a928, 2f32c06c, 5ffd2a3a, bc18bdf4, 09545..."
4,< 0.6,CVE-2020-8bdb,3.0,< 0.6,3.1,965,26,"[d4ec2518, c077c0fd, a0334223, 0cd0e8fc, 8771e..."
...,...,...,...,...,...,...,...,...
995,>= 0.8,CVE-2019-2bd9,8.0,>= 0.8,3.1,460,44,"[6c1ca865, de1d0b52, e927c64b, a97e2254, 3f8b1..."
996,< 0.8,CVE-2019-3908,7.5,< 0.8,2.1,950,73,"[a8d9bc78, 8288986e, a5a68e32, 7848dfb3, e76d2..."
997,< 0.4,CVE-2023-0b12,1.7,< 0.4,2.1,666,12,"[6c010944, 5902ea80, 529e1e8b, babf5c8d, 34977..."
998,< 0.8,CVE-2021-7cb5,1.7,< 0.8,2.1,131,31,"[4835bc4b, f25c943c, 8c3f1730, 233bc084, 3208b..."


In [24]:
df_v3.to_parquet("df_v3.parquet")